In [ ]:
import nibabel as nib
import numpy as np
from pathlib import Path
from radiomics import featureextractor
import pandas as pd
import tempfile
import os
#dynamic insertion
scan_dir = Path(r"C:\Users\cutet\Downloads\PKG - UCSD-PTGBM-v1\Images")
mask_dir = Path(r"C:\Users\cutet\Downloads\PKG - UCSD-PTGBM-v1\masks")

def get_case_id(filename):
    parts = filename.stem.split("_")
    return parts[0] + "_" + parts[1]

def prepare_binary_mask(mask_path):
    mask_img = nib.load(str(mask_path))
    mask_data = mask_img.get_fdata()
    binary_mask = (mask_data > 0).astype(np.uint8)
    temp = tempfile.NamedTemporaryFile(suffix=".nii", delete=False)
    nib.save(nib.Nifti1Image(binary_mask, mask_img.affine), temp.name)
    return temp.name

# Build mask lookup
mask_lookup = {}
for mask_file in mask_dir.glob("*.nii"):
    case_id = get_case_id(mask_file)
    mask_lookup[case_id] = mask_file

# Configure extractor
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.disableAllFeatures()
extractor.enableFeatureClassByName("shape")
#extractor.enableFeatureClassByName("firstorder")
extractor.enableFeatureClassByName("glcm")
extractor.enableFeatureClassByName("glszm")

# Extract features for all pairs
results = []

for scan_file in scan_dir.glob("*.nii"):
    case_id = get_case_id(scan_file)
    if case_id not in mask_lookup:
        continue

    mask_file = mask_lookup[case_id]
    print(f"Processing {case_id}...")

    temp_mask_path = None
    try:
        temp_mask_path = prepare_binary_mask(mask_file)
        feature_dict = extractor.execute(str(scan_file), temp_mask_path)

        numeric_features = {
            k: float(v)
            for k, v in feature_dict.items()
            if k.startswith("original_") and not isinstance(v, str)
        }

        numeric_features["case_id"] = case_id
        results.append(numeric_features)
        print(f"  {case_id} — {len(numeric_features)-1} features extracted")

    except Exception as e:
        print(f"  ERROR on {case_id}: {e}")

    finally:
        if temp_mask_path:
            try:
                os.unlink(temp_mask_path)
            except:
                pass

# Convert to dataframe for visualization
df = pd.DataFrame(results).set_index("case_id")

print(f"\nTotal cases processed: {len(df)}")
print(f"Features per case:     {df.shape[1]}")
print(f"\nFeature names:\n{list(df.columns)}")
print(f"\nFirst case preview:\n{df.iloc[0]}")

Processing UCSD-PTGBM-0002_01...


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


  UCSD-PTGBM-0002_01 — 54 features extracted
Processing UCSD-PTGBM-0005_01...


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


  UCSD-PTGBM-0005_01 — 54 features extracted
Processing UCSD-PTGBM-0005_02...


KeyboardInterrupt: 

In [ ]:
df.to_excel(r"C:\Users\cutet\Downloads\radiomic_features.xlsx")

In [7]:
from sklearn.preprocessing import StandardScaler
import pickle

# Normalize the feature vectors
scaler = StandardScaler()
df_normalized = pd.DataFrame(
    scaler.fit_transform(df),
    index=df.index,
    columns=df.columns
)

# Save the scaler
scaler_path = r"C:\Users\cutet\Desktop\ClinIQ\radiomic_scaler.pkl"
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)

print(f"Scaler saved to: {scaler_path}")
print(f"\nNormalized dataframe shape: {df_normalized.shape}")
print(f"\nFirst case normalized preview:\n{df_normalized.iloc[0]}")

# Save normalized dataframe to excel too
df_normalized.to_excel(r"C:\Users\cutet\Desktop\ClinIQ\radiomic_features_normalized.xlsx")
print("Normalized features saved to excel")

Scaler saved to: C:\Users\cutet\Desktop\ClinIQ\radiomic_scaler.pkl

Normalized dataframe shape: (184, 54)

First case normalized preview:
original_shape_Elongation                          0.769369
original_shape_Flatness                           -0.248101
original_shape_LeastAxisLength                    -0.040597
original_shape_MajorAxisLength                     0.011648
original_shape_Maximum2DDiameterColumn             0.134939
original_shape_Maximum2DDiameterRow               -0.476774
original_shape_Maximum2DDiameterSlice              0.636147
original_shape_Maximum3DDiameter                  -0.275922
original_shape_MeshVolume                          0.510217
original_shape_MinorAxisLength                     0.804270
original_shape_Sphericity                          0.291142
original_shape_SurfaceArea                         0.206338
original_shape_SurfaceVolumeRatio                 -0.991368
original_shape_VoxelVolume                         0.509768
original_glcm_Autocorr

In [4]:
import pandas as pd

df = pd.read_excel(r"C:\Users\cutet\Desktop\ClinIQ\radiomic_features.xlsx", index_col=0)
print(f"Loaded {len(df)} cases, {df.shape[1]} features")

Loaded 184 cases, 54 features


In [ ]:
pip install pyradiomics --user